In [1]:
! pip install flaml

In [ ]:
# baseline_automl_flaml.py
import pandas as pd
import numpy as np
from flaml import AutoML
import os
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")

# ==================== НАСТРОЙКИ ====================
TRAIN_PATH = "train.csv"          # <-- поменяй
TEST_PATH = "test.csv"            # <-- поменяй
TARGET = "target"                 # <-- имя целевой колонки
SUBMISSION_NAME = "flaml_submission.csv"
TIME_BUDGET = 3600 * 4            # 4 часа — обычно хватает даже на большие данные (можно 1800–7200)
TASK = "regression"               # "regression" или "classification"
METRIC = "rmse"                   # для регрессии: rmse, mae, mape; для классификации: roc_auc, log_loss, accuracy и т.д.
SEED = 42
# ====================================================

# Загрузка данных
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

# Если есть id — сохраняем для сабмита
if 'id' in test.columns:
    sample_submission = test[['id']]
else:
    sample_submission = pd.DataFrame({'id': test.index})

y = train[TARGET]
train = train.drop([TARGET], axis=1)

# Авто-очистка: удаляем колонки, которых нет в тесте (иногда бывает)
common_cols = train.columns.intersection(test.columns)
train = train[common_cols]
test = test[common_cols]

print(f"Train shape: {train.shape}, Test shape: {test.shape}")

# ==================== FLAML ====================
automl = AutoML()

settings = {
    "time_budget": TIME_BUDGET,      # секунды
    "metric": METRIC,                # минимизируемая метрика
    "task": TASK,                    # regression / classification
    "log_file_name": "flaml.log",
    "seed": SEED,
    "early_stop": True,
    "verbose": 3,
    "ensemble": True,                # включаем Stacking/Voting автоматически (очень важно!)
    "estimator_list": ['lgbm', 'xgboost', 'catboost', 'rf', 'extra_tree', 'xgb_limitdepth'],
    # можно убрать лишнее, если хочешь быстрее
}

automl.fit(
    X_train=train,
    y_train=y,
    **settings
)

# ==================== ИНФО О ЛУЧШИХ МОДЕЛЯХ ====================
print("\n" + "="*60)
print("Лучшая модель и её метрика на валидации:")
print(f"Best model: {automl.best_estimator}")
print(f"Best config: {automl.best_config}")
print(f"Best {METRIC} on validation: {automl.best_loss:.6f}")
print("="*60)

# Все обученные модели и их скоры
print("\nТоп-10 моделей по валидационному {METRIC}:")
results = automl.best_result  # основная
# Более подробная табличка всех испытанных конфигураций
for i, (estimator, config, val_loss, time) in enumerate(automl.results[:10]):
    print(f"{i+1:2d}. {estimator:15} | {METRIC}: {val_loss:.6f} | time: {time/60:.1f} min")

# Feature importance лучшей модели
print("\nFeature importance (топ-20) лучшей модели:")
try:
    importances = automl.feature_importances_
    if importances is not None:
        fi = pd.Series(importances, index=train.columns).sort_values(ascending=False)
        print(fi.head(20))
except:
    print("   (importance недоступно для этой модели)")

# ==================== ПРЕДСКАЗАНИЯ И САБМИТ ====================
preds = automl.predict(test)

if TASK == "regression":
    sample_submission[TARGET] = preds
else:  # classification — вероятность положительного класса
    sample_submission[TARGET] = preds

sample_submission.to_csv(SUBMISSION_NAME, index=False)
print(f"\nСабмит сохранён: {SUBMISSION_NAME}")
print(f"Предсказания — среднее/медиана: {np.mean(preds):.4f} / {np.median(preds):.4f}")

# Если хочешь локально посчитать RMSE на train (hold-out)
y_pred_train = automl.predict(train)
if TASK == "regression":
    rmse_train = mean_squared_error(y, y_pred_train, squared=False)
    print(f"Train RMSE лучшей модели: {rmse_train:.6f}")